In [ ]:
import numpy as np
from scipy.optimize import curve_fit

from zne import fold_circuit, extrapolate_zne
from circuits import generate_ising_benchmark

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_ibm_runtime import Estimator as AerEstimator

In [ ]:
noise_model = NoiseModel()
error_1q = depolarizing_error(1E-3, 1)
noise_model.add_all_qubit_quantum_error(error_1q, ["x", "y", "z", "h", "s", "t"])
error_2q = depolarizing_error(1E-2, 2)
noise_model.add_all_qubit_quantum_error(error_2q, ["cx"])

n_qubits = 4
qc, observable = generate_ising_benchmark(n_qubits=n_qubits, trotter_steps=2)

lambda_scales = [2*k+1 for k in range(10)]
batches = 1
shots_per_batch = int(1E5)
shots = shots_per_batch * batches
bootstrap_samples = 100

folding_strategy = "global"

target = "probability"

if target == "observable":
    estimator = AerEstimator(
        backend_options={"noise_model": noise_model},
        run_options={"shots": shots_per_batch}
    )
elif target == "probability":
    aer_simulator = AerSimulator(noise_model=noise_model)

print("--- Step 1: Gathering Data ---")
data = {}

for lambda_scale in lambda_scales:
    qc_scaled = fold_circuit(qc, lambda_scale, strategy=folding_strategy)

    if target == "observable":
        result = estimator.run([qc_scaled] * batches, [observable] * batches, optimization_level=0).result()
        values = result.values
        data[lambda_scale] = values

        print(f"Scale λ={lambda_scale}: Mean E = {np.mean(values):.4f}, Variance = {np.var(values):.4f}")
    elif target == "probability":
        qc_scaled.measure_all()
        
        result = aer_simulator.run(qc_scaled, shots=shots, optimization_level=0).result()
        counts = result.get_counts()
        p0 = counts.get("0"*n_qubits, 0)/shots
        data[lambda_scale] = p0
    
        print(f"Scale λ={lambda_scale}: Mean E = {p0:.4f}, Variance = {p0:.4f}")

raw_expvals = [np.mean(values) for values in data.values()]

print("\n--- Step 2: Extrapolating on data ---")
for extrapolation_method in ["linear", "bayesian_linear", "richardson", "exponential", "gaussian_process"]:
    zero_noise_exp_val = extrapolate_zne(lambda_scales, raw_expvals, method=extrapolation_method)
    
    bootstrapped_znes = []
    for _ in range(bootstrap_samples):
        noise = np.random.normal(0, 1/np.sqrt(shots), len(lambda_scales))
        resampled_exp_vals = np.array(raw_expvals) + noise
        
        bootstrapped_znes.append(extrapolate_zne(lambda_scales, raw_expvals, method=extrapolation_method))
    
    std_err = np.std(bootstrapped_znes) if bootstrap_samples > 0 else 0.0
    
    print(f"\nExtrapolated Zero-Noise Value via {extrapolation_method}: {zero_noise_exp_val:.4f} ± {std_err:.4f}")